# FastText Embedding + Random Forest ile Metin Sınıflandırma
**Veri Seti:** 20 Newsgroups (4 Kategori)

**Kategoriler:** sci.electronics, sci.space, comp.windows.x, rec.motorcycles

**Embedding:** cc.en.300.bin (Facebook Pre-trained FastText)

**Sınıflandırıcı:** Random Forest

## 1. Kurulum & Model İndirme (Colab İçin)

In [ ]:
# Colab'te fasttext kütüphanesini kur
!pip install fasttext -q

In [ ]:
# cc.en.300.bin modelini indir (~4.5 GB sıkıştırılmış)
# Bu hücre 5-10 dk sürebilir
!wget -q --show-progress https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
!gunzip cc.en.300.bin.gz
print("Model başarıyla indirildi!")

## 2. Kütüphanelerin Import Edilmesi

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import re
import numpy as np
import fasttext
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             precision_score, recall_score, f1_score)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

print("Tüm kütüphaneler yüklendi!")

## 3. FastText Modelini Yükle

In [ ]:
ft_model = fasttext.load_model('cc.en.300.bin')

print(f"Model yüklendi!")
print(f"Vektör boyutu: {ft_model.get_dimension()}")
print(f"Kelime sayısı: {len(ft_model.words)}")

## 4. Veri Setini İndir (4 Kategori)

In [ ]:
categories = [
    'sci.electronics',
    'sci.space',
    'comp.windows.x',
    'rec.motorcycles'
]

newsgroups_data = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

print(f"Toplam doküman sayısı: {len(newsgroups_data.data)}")
print(f"Kategoriler: {newsgroups_data.target_names}")
print(f"Etiket dağılımı: {dict(zip(newsgroups_data.target_names, np.bincount(newsgroups_data.target)))}")

## 5. Metin Ön İşleme (Preprocessing)

TF-IDF'ten farkı: `' '.join(words)` yapmıyoruz, kelime listesi olarak bırakıyoruz.

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = word_tokenize(text)

    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]

    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words and len(word) > 1]

    # FastText'te join YAPMIYORUZ - liste döndürüyoruz
    return words

In [ ]:
cleaned_data = [preprocess_text(doc) for doc in newsgroups_data.data]

print(f"Örnek doküman (ilk 15 kelime): {cleaned_data[0][:15]}")
print(f"Tip: {type(cleaned_data[0])}  ← Liste! String değil!")

## 6. Doküman Vektörleri Oluştur (Mean Pooling)

In [ ]:
def document_vector(word_list, model):
    if len(word_list) == 0:
        return np.zeros(model.get_dimension())

    vectors = [model.get_word_vector(word) for word in word_list]
    return np.mean(vectors, axis=0)


print("Doküman vektörleri oluşturuluyor...")
X = np.array([document_vector(doc, ft_model) for doc in cleaned_data])
y = newsgroups_data.target

print(f"Feature matrix şekli: {X.shape}")
print(f"  → {X.shape[0]} doküman, her biri {X.shape[1]} boyutlu vektör")

## 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train seti: {X_train.shape[0]} doküman")
print(f"Test seti:  {X_test.shape[0]} doküman")

## 8. Random Forest ile Model Eğitimi

In [ ]:
clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Model eğitimi tamamlandı!")

## 9. Sonuçlar

In [ ]:
print("=" * 60)
print("SONUÇLAR")
print("=" * 60)

print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='macro'):.4f}")

In [ ]:
print("\nDetaylı Classification Report:")
print("=" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=newsgroups_data.target_names
))